# 第四轮：把最佳起点写进训练损失
训练和评测都使用原始输入的频带帧作为复述模板。`aligned` 对最后一个有声输入帧之后的所有允许起点计算逐帧损失，用 soft-min 近似选择最佳起点；`fixed` 使用同一模板，但固定在旧数据随机指定的起点。两模型架构、初始化、数据种子、步数和评测样本相同。没有向模型提供内部断句标志。

步骤：① 在 Colab 选择 GPU；② 获取 GitHub 代码；③ 挂载 Drive；④ 运行快速检查；⑤ 依次训练 aligned 与 fixed；⑥ 查看结果；⑦ 将 `MyDrive/Parrot/trial4_results.zip` 回传。每 100 步将日志写到 Drive；如运行中断，请回传已生成的日志。


In [ ]:
from pathlib import Path
import subprocess
repo = Path('/content/Parrot')
if not repo.exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/DoctorAwe/Parrot.git', str(repo)], check=True)
else:
    subprocess.run(['git', '-C', str(repo), 'pull', '--ff-only'], check=True)
assert (repo / 'trial4.py').is_file()
commit = subprocess.check_output(['git', '-C', str(repo), 'rev-parse', '--short', 'HEAD'], text=True).strip()
print('代码版本:', commit)
%cd /content/Parrot


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
drive_root = Path('/content/drive/MyDrive/Parrot')
root = drive_root / 'trial4_results'
root.mkdir(parents=True, exist_ok=True)
print('结果目录:', root)


快速检查只验证前向、反向、对齐损失和结果导出，不用于判断模型性能。


In [ ]:
!python trial4.py --objective aligned --smoke --batch 4 --out /content/Parrot/trial4_smoke_aligned
!python trial4.py --objective fixed --smoke --batch 4 --out /content/Parrot/trial4_smoke_fixed


先训练使用最佳时间平移损失的模型。


In [ ]:
!python trial4.py --objective aligned --steps 1800 --batch 24 --eval 200 --out {root / 'aligned'}


再训练固定起点对照模型；请勿跳过，否则无法判断训练损失修改的收益。


In [ ]:
!python trial4.py --objective fixed --steps 1800 --batch 24 --eval 200 --out {root / 'fixed'}


In [ ]:
import json
from IPython.display import Audio, display
for objective in ('aligned', 'fixed'):
    path = root / objective / 'metrics.json'
    report = json.loads(path.read_text())
    report['git_commit'] = commit
    path.write_text(json.dumps(report, indent=2))
    print(objective, report['conditions']['0.15']['overall'])
    print('重复符号:', report['conditions']['0.15']['repeated'])
    display(Audio(filename=str(root / objective / 'model.wav')))


In [ ]:
import shutil
archive = shutil.make_archive(str(drive_root / 'trial4_results'), 'zip', root_dir=str(drive_root), base_dir='trial4_results')
print('请回传:', archive)
